# 文档切分器 Text Splitters

文档加载后通常得到较大的 `Document`。切分（Chunking）负责把它转换成适合索引、检索和生成的较小 `Document`。

切分的主要目标不是单纯满足模型上下文长度，而是在以下因素之间取得平衡：

- **可检索性**：一个 Chunk 尽量只表达一个相对集中的主题。
- **上下文完整性**：答案所需信息不要被不合理地拆散。
- **长度约束**：适配 Embedding 模型和生成模型的输入限制。
- **成本与吞吐量**：Chunk 越多，Embedding、存储和检索成本越高。
- **可追溯性**：保留 source、页码、标题层级和起始位置等 metadata。

## 常见切分策略

| 策略 | 主要依据 | 适合场景 |
| --- | --- | --- |
| 分隔符切分 | 换行、空格或自定义符号 | 格式稳定的简单文本 |
| 递归切分 | 段落 → 行 → 词 → 字符 | 通用文本，官方推荐起点 |
| Token 切分 | Tokenizer 计算的 Token 数 | 严格适配模型输入限制 |
| 文档结构切分 | Markdown 标题、HTML 标签、JSON 对象、代码结构 | 具有明确层级的内容 |
| 语义分块 | 相邻句子的 Embedding 距离 | 主题变化明显、结构标记不足的文本 |
| 结构优先、长度兜底 | 先按标题/元素，再拆分过长块 | 生产项目中的常见组合 |

`RecursiveCharacterTextSplitter` 不执行 Embedding 或语义分析。它只是优先保留段落、句子附近的自然分隔符，再以长度作为约束。

## Chunk 不一定等于最终返回内容

朴素 RAG 经常直接索引并返回同一个 Chunk，但这不是强制规则：

```text
小块或摘要用于索引和匹配
          ↓
命中后根据 ID 返回更大的父文档、原始表格或图片
```

因此切分阶段要同时考虑：索引单元、召回单元、最终上下文和证据引用。Multi-vector、parent-child、small-to-big RAG 会在后续检索策略章节展开。

In [2]:
from pathlib import Path
from langchain_core.documents import Document

CURRENT_DIR = Path.cwd().resolve()
RAG_DIR = next(
    (path for path in (CURRENT_DIR, *CURRENT_DIR.parents) if path.name == "6-LangChain中的RAG"),
    CURRENT_DIR / "系统学习" / "6-LangChain中的RAG",
)
source_path = RAG_DIR / "knowledge.txt"
document = Document(
    page_content=source_path.read_text(encoding="utf-8"),
    metadata={"source": str(source_path)},
)

print(f"原文字符数：{len(document.page_content)}")
print(document.metadata)


原文字符数：5323
{'source': 'E:\\AI_Projects\\LearnProjects\\Agent\\LangChain_Learn\\系统学习\\6-LangChain中的RAG\\knowledge.txt'}


## 学习顺序

先掌握 `CharacterTextSplitter`、`RecursiveCharacterTextSplitter` 和 Token 切分，再学习 Markdown、HTML、JSON、代码等结构切分。最后比较语义分块、Docling 结构切分和混合策略。切分策略没有脱离数据集的统一最优答案，应通过检索评估选择。